Query 1

Query Statement: Find the rank, dense rank, and row number of employees based on their HireDate, grouped by department.

Tables needed to solve: HumanResources.Employee, HumanResources.EmployeeDepartmentHistory

Use of the query: To know the seniority of employees within each department based on when they were hired.

In [3]:
 Use AdventureWorks2019;

SELECT 
    EmpData.DepartmentName,
    EmpData.BusinessEntityID,
    EmpData.JobTitle,
    EmpData.HireDate,
    RANK() OVER(PARTITION BY EmpData.DepartmentName ORDER BY EmpData.HireDate) AS HireRank,
    DENSE_RANK() OVER(PARTITION BY EmpData.DepartmentName ORDER BY EmpData.HireDate) AS HireDenseRank,
    ROW_NUMBER() OVER(PARTITION BY EmpData.DepartmentName ORDER BY EmpData.HireDate) AS HireRowNum
FROM (
    SELECT 
        E.BusinessEntityID,
        E.HireDate,
        E.JobTitle,
        D.Name AS DepartmentName
    FROM HumanResources.Employee E
    JOIN HumanResources.EmployeeDepartmentHistory DH
        ON E.BusinessEntityID = DH.BusinessEntityID
    JOIN HumanResources.Department D
        ON D.DepartmentID = DH.DepartmentID
    WHERE DH.EndDate IS NULL -- Only currently active department assignments
) AS EmpData
ORDER BY EmpData.DepartmentName, EmpData.HireDate;

-- Explanation:
-- 1. A derived table (subquery in FROM) gathers relevant employee info along with department.
-- 2. Filters current employees only (EndDate IS NULL).
-- 3. Window functions are applied on the outer query for ranking.

: Msg 10347, Level 16, State 1, Procedure Employee, Line 1
Common Language Runtime(CLR) is not enabled on this instance.

: Msg 474, Level 16, State 0, Line 3
Unable to load the computed column definitions for table "HumanResources.Employee".

Total execution time: 00:00:00.008

Query 2

Query Statement: Find the number of addresses in each city and rank them by how many addresses each city has, in descending order.

  

Tables needed to solve: Person.Address

  

Use of the query: To know which cities they have the most address entries in—useful for planning logistics, deliveries, or setting up local offices.

In [ ]:
USE AdventureWorks2019;
SELECT 
    City,
    COUNT(*) AS TotalAddresses,
    RANK() OVER(ORDER BY COUNT(*) DESC) AS CityRank,
    DENSE_RANK() OVER(ORDER BY COUNT(*) DESC) AS CityDenseRank,
    ROW_NUMBER() OVER(ORDER BY COUNT(*) DESC) AS CityRowNumber
FROM Person.Address
GROUP BY City
ORDER BY TotalAddresses DESC;

-- Explanation:
-- 1. We group the addresses by City and count how many addresses are in each city.
-- 2. Window functions (RANK, DENSE_RANK, ROW_NUMBER) are applied to rank cities based on address count.
-- 3. Results are sorted by the total number of addresses per city in descending order.

Query 3

  

Query Statement: Use the ROW\_NUMBER() window function to assign a unique row number to each phone number, partitioned by the person.

  

Tables needed to solve: Person.Person, Person.PersonPhone

  

Use of the query: To identify and number phone numbers for each person (e.g., first phone, second phone) in cases where the data needs to be sorted or ranked.

In [ ]:
USE AdventureWorks2019;

SELECT 
    p.BusinessEntityID,
    p.FirstName,
    p.LastName,
    ph.PhoneNumber,
    ROW_NUMBER() OVER (PARTITION BY p.BusinessEntityID ORDER BY ph.PhoneNumber) AS PhoneNumberRank
FROM 
    Person.Person p
LEFT JOIN 
    Person.PersonPhone ph ON p.BusinessEntityID = ph.BusinessEntityID;

-- Explanation:
-- 1. We join Person with PersonPhone on BusinessEntityID.
-- 2. ROW_NUMBER() assigns a unique number to each phone number for a person, starting from 1.
-- 3. PARTITION BY p.BusinessEntityID ensures the numbering restarts for each person.
-- 4. ORDER BY ph.PhoneNumber makes sure phone numbers are sorted (so row numbers are sequential based on phone number).
-- 5. LEFT JOIN ensures persons without phone numbers are included.

Query 4

  

Query Statement: Transform production transaction history data into a format where each transaction type (like 'S' for Sale, 'P' for Purchase) is represented as a column, showing the total quantity of products sold and purchased.

  

Tables needed to solve: Production.TransactionHistory

  

Use of the query: This query can be used to summarize transaction history data by showing the total quantity of products sold and purchased in separate columns for each product.

In [ ]:
USE AdventureWorks2019;

SELECT 
    ProductID,
    ISNULL([S], 0) AS SalesQuantity,   -- Sales quantity, defaults to 0 if no sales
    ISNULL([P], 0) AS PurchaseQuantity  -- Purchase quantity, defaults to 0 if no purchase
FROM 
    (SELECT ProductID, TransactionType, Quantity
     FROM Production.TransactionHistory) AS SourceTable
PIVOT
    (SUM(Quantity) FOR TransactionType IN ([S], [P])) AS PivotTable;


-- 1. First, we select ProductID, TransactionType, and Quantity from the source table (TransactionHistory).
-- 2. Then, we use PIVOT to convert the TransactionType ('S' for Sales and 'P' for Purchase) into columns.
-- 3. The SUM() function calculates the total quantity for each transaction type per product.
-- 4. ISNULL ensures that if no transactions exist for a given type (Sale or Purchase), a 0 is shown instead.

Query 5

Query Statement: Rank products based on their ListPrice, and compare how the rankings differ when handling ties.

Tables needed to solve: Production.Product

  

Use of the query: To determine the ranking of products based on their price, and also for pricing strategies and analyzing how multiple products with the same price are treated in terms of ranking.

In [ ]:
USE AdventureWorks2019;

SELECT 
    ProductID, 
    Name, 
    ListPrice,
    RANK() OVER (ORDER BY ListPrice DESC) AS RankByPrice,  -- Rank products based on ListPrice, handling ties with gaps
    DENSE_RANK() OVER (ORDER BY ListPrice DESC) AS DenseRankByPrice -- Rank products based on ListPrice, handling ties without gaps
FROM 
    Production.Product
WHERE 
    ListPrice IS NOT NULL
ORDER BY 
    RankByPrice;


-- 1. The RANK() function assigns ranks to products based on their ListPrice. If two products have the same price, they receive the same rank, 
--    but there will be a gap in the ranking for the next product (e.g., if two products are tied at rank 1, the next product gets rank 3).
-- 2. The DENSE_RANK() function works similarly, but it does not leave gaps in the ranking. If two products are tied at rank 1, 
--    the next product will be ranked 2 instead of 3.
-- 3. Both rankings are ordered by `ListPrice` in descending order to give the highest-priced products the highest rank.

Query 6

  

 Query Statement: Calculate the total quantity ordered and received for each product, and then rank products based on the total quantity received, with the highest quantity receiving the top rank.

  

Tables needed to solve: Purchasing.PurchaseOrderDetail

  

Use of the query: This query can be useful for understanding which products have the highest quantity received from purchase orders. It helps businesses track the products that are being stocked the most and prioritize those that are receiving the highest quantity, which could assist in inventory and purchasing decisions.

In [ ]:
USE AdventureWorks2019;

WITH ProductReceived AS (
    -- CTE to calculate the total received quantity for each product
    SELECT 
        ProductID, 
        SUM(ReceivedQty) AS TotalReceivedQty
    FROM 
        Purchasing.PurchaseOrderDetail
    GROUP BY 
        ProductID
)
SELECT 
    pr.ProductID,
    p.Name,  -- Assuming you have a 'Product' table that stores product names
    pr.TotalReceivedQty,
    RANK() OVER (ORDER BY pr.TotalReceivedQty DESC) AS RankByReceivedQty
FROM 
    ProductReceived pr
JOIN 
    Production.Product p ON pr.ProductID = p.ProductID
ORDER BY 
    RankByReceivedQty;
    

-- 1. The CTE (Common Table Expression) 'ProductReceived' calculates the total quantity of items received for each product.
-- 2. We use the RANK() function to assign ranks to products based on the total quantity received, where the product with the highest quantity
--    received will have the rank 1. If multiple products have the same quantity received, they will share the same rank.
-- 3. The result shows the ProductID, the product's name, the total quantity received, and the rank for each product based on the received quantity.
-- 4. We join the CTE with the Product table to get the product name alongside the total received quantity.

Query 7

  

Query Statement: Retrieve the SalesTaxRateID, StateProvinceID, TaxType, TaxRate, Name, and calculates the rank of each state's tax rate.

  

Tables needed to solve: Sales.SalesTaxRate

  

Use of the query: To analyze tax rates by different tax types across various states or provinces.

In [ ]:
USE AdventureWorks2019;

SELECT 
    str.SalesTaxRateID,         
    str.StateProvinceID,        
    str.TaxType,                
    str.TaxRate,        
    str.Name,                   
    RANK() OVER (PARTITION BY str.TaxType ORDER BY str.TaxRate DESC) AS TaxRateRank
FROM 
    Sales.SalesTaxRate str     
ORDER BY  
    TaxRateRank;               -- Orders the result by the calculated rank

Query 8

Query Statement: Rank the sales order details by the total line amount (LineTotal) for each product and special offer. Assign ranks within each product group and display the top-ranked order for each product-special offer combination.

  

Tables needed to solve: Sales.SalesOrderDetail

  

Use of the query: This query can be used in sales reporting or order management systems to identify and prioritize the highest-value orders per product and special offer.

In [ ]:
USE AdventureWorks2019;

SELECT 
    sod.ProductID,                -- Group by product
    sod.SpecialOfferID,           -- Group by special offer
    SUM(sod.LineTotal) AS TotalSales,  -- Sum the line total for each group
    COUNT(sod.SalesOrderDetailID) AS TotalOrders, -- Count orders for each group
    GROUPING_ID(sod.ProductID, sod.SpecialOfferID) AS GroupingLevel -- Shows which level the aggregation is at
FROM 
    Sales.SalesOrderDetail sod
GROUP BY 
    sod.ProductID, 
    sod.SpecialOfferID
ORDER BY 
    GroupingLevel DESC, TotalSales DESC;


-- 1. The SUM() function calculates the total sales for each group of Product and SpecialOffer.
-- 2. COUNT() is used to count the number of orders for each product-special offer combination.
-- 3. GROUPING_ID() returns a value indicating the level of grouping:
--    - If both ProductID and SpecialOfferID are included in the group, the value will be 0 (no columns are NULL).
--    - If only one of the columns (ProductID or SpecialOfferID) is included, it will return a non-zero value indicating that the other column is NULL.
-- 4. The results are ordered by GroupingLevel (showing the hierarchy of the grouping) and TotalSales.

Query 9

  

Query Statement: Display the order quantity and the difference in order quantity between consecutive orders for each product. Also, show the subsequent order's due date and calculate the lead time for each order.

  

Tables needed to solve: Purchasing.PurchaseOrderDetail, Purchasing.PurchaseOrderHeader

  

Use of the query: Helps in tracking the change in order quantities over time for each product.

In [ ]:
USE AdventureWorks2019;

SELECT 
    pod.ProductID,                                    
    pod.OrderQty,                                       
    LAG(pod.OrderQty) OVER (PARTITION BY pod.ProductID ORDER BY poh.OrderDate) AS PreviousOrderQty, 
    LEAD(pod.OrderQty) OVER (PARTITION BY pod.ProductID ORDER BY poh.OrderDate) AS NextOrderQty,      
    DATEDIFF(DAY, poh.OrderDate, LEAD(poh.OrderDate) OVER (PARTITION BY pod.ProductID ORDER BY poh.OrderDate)) AS LeadTimeDays, 
	poh.OrderDate                                        
FROM 
    Purchasing.PurchaseOrderDetail pod
JOIN 
    Purchasing.PurchaseOrderHeader poh 
    ON pod.PurchaseOrderID = poh.PurchaseOrderID 
ORDER BY 
    pod.ProductID, poh.OrderDate; 




-- 1. The query selects the ProductID and OrderQty from the PurchaseOrderDetail table for each product.
-- 2. LAG() and LEAD() functions are used to retrieve the previous and next order quantities for the same product, partitioned by ProductID and ordered by OrderDate.
-- 3. DATEDIFF() calculates the lead time (in days) between the current order and the next order for the same product.
-- 4. The result is ordered by ProductID and OrderDate to ensure data is organized by product and chronological order.
-- 5. This query helps analyze order patterns, track product demand over time, and identify lead times between consecutive orders.

Query 10

  

Query Statement: Display the sales order ID, product ID, order quantity, and the rank of each order item based on the order quantity in descending order for each sales order.

  

Tables needed to solve: Sales.SalesOrderHeader, Sales.SalesOrderDetail

  

Use of the query: Helps to rank products based on the order quantity within each sales order, allowing identification of the most significant items in terms of quantity for each order

In [ ]:
USE AdventureWorks2019;

SELECT 
    soh.SalesOrderID,                                           
    soh.OrderDate,                                              
    sod.ProductID,                                              
    sod.OrderQty,                                               
    ROW_NUMBER() OVER (PARTITION BY soh.SalesOrderID ORDER BY sod.OrderQty DESC) AS OrderRank
    -- Assigns a row number to each order item within the same sales order based on the quantity (largest first)
FROM 
    Sales.SalesOrderDetail sod
JOIN 
    Sales.SalesOrderHeader soh
    ON sod.SalesOrderID = soh.SalesOrderID                      -- Joining sales order header with sales order detail
ORDER BY 
    soh.SalesOrderID, OrderRank;                                 -- Ordering the results by SalesOrderID and the generated row number